# Lichess variants extract: the same players across different games (free, on Google Colab)

Picks players who play **standard chess and at least one variant** (Chess960, crazyhouse, atomic, antichess) in the same month, then computes the same per-move measures for **each game type separately** (each type has its own memory of past positions). The same secret key gives each player the **same anonymous code in every file**, so traits can be compared across games within a person.

- Nothing is downloaded to your own computer; raw games are streamed. Only small anonymised files are saved to your Google Drive (`compression-learner-data/lichess_variants/`).
- **Secret key:** the same `LICHESS_ID_SALT` as for the Lichess run (key icon in the left sidebar). The run stops if the key's fingerprint differs from the saved one.
- **Engine check:** Stockfish for standard chess and Chess960 only (it cannot play the other variants).
- **First** run with `TEST_RUN = True` (about 20–30 min). It prints how many players qualify, which is the main thing to check. Then set `TEST_RUN = False` for the full run (several hours; resumable: if it stops, just Run all again).
- **How to run:** Runtime → Run all, allow Drive access.

Code: https://github.com/saurabh9729/compression-learner-data (`pipelines/lichess/`)

In [ ]:
# Settings
TEST_RUN = True
MONTHS = ["2018-06", "2018-07"]               # players are picked from the first month
VARIANTS = ["chess960", "crazyhouse", "atomic", "antichess"]
MIN_STANDARD = 50       # standard games (any speed) in the first month
MIN_VARIANT = 30        # games in at least one of the variants in the first month
N_PLAYERS = 400         # random sample of eligible players
ENGINE_N = 10_000       # positions checked by Stockfish, per game type (standard, chess960)
ENGINE_DEPTH = 12
MAX_GAMES = 1_000_000 if TEST_RUN else 0      # test: only the first 1M standard games per month
if TEST_RUN:
    MIN_STANDARD, MIN_VARIANT, ENGINE_N = 10, 10, 300
TYPES = ["standard"] + VARIANTS

In [ ]:
# Install tools and fetch the pipeline code from the public repo
!apt-get -qq install -y zstd stockfish > /dev/null
!pip -q install chess pyarrow
import os, getpass
BASE = "https://raw.githubusercontent.com/saurabh9729/compression-learner-data/main/pipelines/lichess"
os.makedirs("/content/work", exist_ok=True); os.chdir("/content/work")
for f in ["filter_lichess.py", "select_players_variants.py", "lichess_pipeline.py", "check_no_names.py", "engine_eval.py"]:
    !curl -sSfLO {BASE}/{f}
!ls -l *.py

In [ ]:
# Secret key (same as the Lichess run) + fingerprint check, so codes match across all files
import hashlib, subprocess
from google.colab import drive
drive.mount("/content/drive")
root = "/content/drive/MyDrive/compression-learner-data"
dest = f"{root}/lichess_variants"
os.makedirs(dest, exist_ok=True)
key = None
try:
    from google.colab import userdata
    key = userdata.get("LICHESS_ID_SALT")
except Exception:
    pass
if not key:
    key = getpass.getpass("Secret key: ")
assert len(key) >= 12, "use a longer key (12+ characters)"
fp = hashlib.pbkdf2_hmac("sha256", key.encode(), b"lichess-key-check", 200_000).hex()[:6]
fp_file = f"{root}/lichess/key_fingerprint.txt"
if os.path.exists(fp_file):
    saved = open(fp_file).read().strip()
    assert fp == saved, f"STOP: not the same key as the Lichess run (fingerprint {fp}, saved {saved})"
    print("key check: same key as the Lichess run ✓", fp)
else:
    print("no saved fingerprint found; this key's fingerprint is", fp)
os.environ["LICHESS_ID_SALT"] = key

def url(t, m):
    return (f"https://database.lichess.org/standard/lichess_db_standard_rated_{m}.pgn.zst" if t == "standard"
            else f"https://database.lichess.org/{t}/lichess_db_{t}_rated_{m}.pgn.zst")
for t in TYPES:
    for m in MONTHS:
        code_ = subprocess.run(["curl", "-sIL", "-o", "/dev/null", "-w", "%{http_code}", url(t, m)], capture_output=True, text=True).stdout
        print(f"{t:11s} {m}: {'ok' if code_ == '200' else 'NOT FOUND (' + code_ + ')'}")

In [ ]:
# Checkpoints in your private Drive: finished steps are skipped when you Run all again
import shutil
TAG = "_".join(MONTHS) + ("_test" if TEST_RUN else "")
WORK = f"{dest}/work_{TAG}"
os.makedirs(WORK, exist_ok=True)
def have(f): return os.path.exists(os.path.join(WORK, f))
def save(f): shutil.copy(f, WORK)
def restore(f): shutil.copy(os.path.join(WORK, f), f)
def sh(cmd):
    # run a step; show its key messages, and the full error text if it fails
    r = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)
    out = r.stdout + r.stderr
    keys = ("skipped", "games skipped", "moves:", "players", "eligible", "selected", "  ...", "similar-position",
            "engine rows", "no moves", "checked", "ok:", "FAIL", "games to process")
    print("\n".join(l for l in out.splitlines() if l.startswith(keys)))
    if r.returncode:
        print("---- error output ----\n" + out[-4000:])
        raise RuntimeError("this step failed; the error text is above")
print("checkpoints so far:", sorted(os.listdir(WORK)) or "none")

In [ ]:
# Pass 1: count games per player per game type in the FIRST month, then pick players
if have("players.txt"):
    restore("players.txt"); print("players restored from checkpoint")
else:
    for t in TYPES:
        mg = MAX_GAMES if t == "standard" else 0
        print("counting", t)
        sh(f"curl -sL {url(t, MONTHS[0])} | zstdcat 2>/dev/null | python filter_lichess.py --count counts_{t}.csv --max-games {mg}")
    vargs = " ".join(f"--variant {v}=counts_{v}.csv" for v in VARIANTS)
    sh(f"python select_players_variants.py --standard counts_standard.csv {vargs} "
       f"--min-standard {MIN_STANDARD} --min-variant {MIN_VARIANT} --n {N_PLAYERS} --out players.txt")
    save("players.txt")

In [ ]:
# Pass 2: keep only their games (with clock times), month after month, one file per game type
for t in TYPES:
    f = f"filtered_{t}.pgn"
    if have(f + ".zst"):
        restore(f + ".zst"); sh(f"zstd -d -q -f {f}.zst -o {f}"); print(t, "restored from checkpoint")
        continue
    sh(f"rm -f {f}")
    for m in MONTHS:
        mg = MAX_GAMES if t == "standard" else 0
        print("streaming", t, m)
        sh(f"curl -sL {url(t, m)} | zstdcat 2>/dev/null | python filter_lichess.py --extract players.txt --max-games {mg} >> {f}")
    sh(f"zstd -q -T0 -3 -f {f} -o {f}.zst"); save(f + ".zst")
for t in TYPES:
    n = int(subprocess.run(["bash", "-c", f"grep -c '^\\[Event ' filtered_{t}.pgn || true"], capture_output=True, text=True).stdout.strip() or 0)
    print(f"{t:11s} games kept: {n:,}")

In [ ]:
# Per-move measures for each game type separately (own memory per type; same anonymous codes)
OUT = "moves_" + TAG
for t in TYPES:
    o = f"{OUT}_{t}"
    if have(o + ".parquet") and have(o + "_players.csv"):
        restore(o + ".parquet"); restore(o + "_players.csv"); print(t, "restored from checkpoint"); continue
    if os.path.getsize(f"filtered_{t}.pgn") == 0:
        print(t, "no games, skipped"); continue
    print("processing", t)
    sh(f"python lichess_pipeline.py filtered_{t}.pgn --players players.txt --out {o}.parquet")
    save(o + ".parquet"); save(o + "_players.csv")
    sh(f"python check_no_names.py players.txt {o}.parquet {o}_players.csv")

In [ ]:
# Engine check (standard chess and Chess960 only)
for t in ["standard", "chess960"]:
    o = f"{OUT}_{t}"
    if not os.path.exists(o + ".parquet"):
        continue
    if have(o + "_engine.parquet"):
        restore(o + "_engine.parquet"); print(t, "engine restored from checkpoint"); continue
    sh(f"python engine_eval.py {o}.parquet --out {o}_engine.parquet --n {ENGINE_N} --depth {ENGINE_DEPTH}")
    save(o + "_engine.parquet")
    sh(f"python check_no_names.py players.txt {o}_engine.parquet")

In [ ]:
# Save only the anonymised files to Google Drive (only if the safety check passes).
# Big files are split by player into parts of about 25 MB, so they can be uploaded to a chat.
import math, zlib, glob
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
files = sorted(glob.glob(f"{OUT}_*.parquet") + glob.glob(f"{OUT}_*_players.csv"))
ok = subprocess.run(["python", "check_no_names.py", "players.txt", *files]).returncode == 0
assert ok, "safety check failed: nothing saved"
saved = []
for f in files:
    size_mb = os.path.getsize(f) / 1e6
    if f.endswith(".csv") or f.endswith("_engine.parquet") or size_mb <= 25:
        shutil.copy(f, dest); saved.append(f); continue
    n_parts = math.ceil(size_mb / 25)
    t = pq.read_table(f)
    part_col = pa.array([zlib.crc32(p.encode()) % n_parts for p in t.column("player").to_pylist()])
    for k in range(n_parts):
        name = f.replace(".parquet", f"_part{k + 1}of{n_parts}.parquet")
        pq.write_table(t.filter(pc.equal(part_col, k)), name)
        shutil.copy(name, dest); saved.append(name)
print("saved to Drive:", dest)
for f in saved: print("  ", f, f"{os.path.getsize(os.path.join(dest, os.path.basename(f))) / 1e6:.1f} MB")
shutil.rmtree(WORK, ignore_errors=True)   # checkpoints hold raw games with usernames
print("checkpoints removed")